# Import and Connect to DB

In [3]:
import os
import sys
import time
from datetime import datetime
from typing import Tuple
from loguru import logger
from pymongo import MongoClient
from pymongo.database import Database
from concurrent.futures import ThreadPoolExecutor

# PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))))
PROJECT_ROOT = "/home/ubuntu/projects/AI/git/users/giangnv/v03/V03-Sync-Services"
sys.path.append(PROJECT_ROOT)

from constants import MongoDBConfig, MigrateConfig, MongoDBCollectionConfig
from core.v03.segments_classifier import classify_segment
# from jobs.law.mirgrate_classification import post_process_classify

# Constants
BATCH_SIZE = 1000
MAX_WORKERS = 1

def connect_to_databases() -> Tuple[MongoClient, Database]:
    """Connect to MongoDB core_db.

    Returns:
        Tuple[MongoClient, Database]: MongoDB client and core database.

    Raises:
        Exception: If connection fails.
    """
    try:
        client = MongoClient(
            host=MongoDBConfig.HOST,
            port=MongoDBConfig.PORT,
            username=MongoDBConfig.USERNAME,
            password=MongoDBConfig.PASSWORD
        )
        core_db = client[MigrateConfig.MIGRATE_CORE_DB]
        return client, core_db
    except Exception as e:
        logger.error(f"Failed to connect to MongoDB: {e}")
        raise


logger.info("Connecting to MongoDB...")
_, core_db = connect_to_databases()


2026-02-03 08:39:17.052 | INFO     | __main__:<module>:46 - Connecting to MongoDB...


In [5]:
document_collection = core_db[MongoDBCollectionConfig.LAW_DOCUMENT_COLLECTION_NAME]                
articles_collection = core_db[MongoDBCollectionConfig.LAW_ARTICLE_COLLECTION_NAME]
articles_class_collection = core_db[MongoDBCollectionConfig.LAW_ARTICLE_CLASS_COLLECTION_NAME]

# document_collection.create_index([('doc_id', 1)])    
# articles_collection.create_index([('article_id', 1)])
# articles_class_collection.create_index([('article_id', 1)])

# Export Schema

In [7]:
from pymongo import MongoClient


def get_db_schema():
    schema = {}
    for collection_name in core_db.list_collection_names():
        # Lấy một bản ghi bất kỳ để suy luận cấu trúc
        sample = core_db[collection_name].find_one()
        if sample:
            schema[collection_name] = {k: str(type(v)) for k, v in sample.items()}
    return schema

In [8]:
schemas = get_db_schema()

In [ ]:
import json

# Giả sử 'schemas' là dictionary chứa thông tin cấu trúc các collection
for collection_name, schema in schemas.items():
    file_name = f"./data/{collection_name}.json"
    
    try:
        with open(file_name, 'w', encoding='utf-8') as f:
            # indent=4 giúp file JSON dễ đọc (scannable)
            # ensure_ascii=False để hiển thị đúng ký tự tiếng Việt pháp lý
            json.dump(schema, f, indent=4, ensure_ascii=False)
            
        print(f"✅ Đã xuất schema cho collection '{collection_name}' thành công -> {file_name}")
    except Exception as e:
        print(f"❌ Lỗi khi xuất file {file_name}: {e}")

print("="*50)
print("Hoàn tất quá trình trích xuất Data Schema.")

Collection Name: law_articles
{'_id': "<class 'bson.objectid.ObjectId'>", 'article_id': "<class 'str'>", 'doc_id': "<class 'str'>", 'article_title': "<class 'str'>", 'article_content': "<class 'str'>", 'article_order_index': "<class 'int'>", 'created_date': "<class 'str'>", 'created_by': "<class 'str'>", 'last_modified': "<class 'str'>", 'last_modified_by': "<class 'str'>", 'part': "<class 'str'>", 'chapter': "<class 'str'>", 'section': "<class 'str'>", 'sub_section': "<class 'str'>", 'article_class_status': "<class 'NoneType'>", 'MigrateConfig': "<class 'dict'>", 'article_effective_date': "<class 'str'>", 'article_effective_status': "<class 'str'>", 'article_expiry_date': "<class 'str'>", 'decree_status_id': "<class 'str'>"}
Collection Name: law_positions
{'_id': "<class 'bson.objectid.ObjectId'>", 'position_id': "<class 'str'>", 'position_name': "<class 'str'>", 'created_by': "<class 'str'>", 'created_at': "<class 'str'>", 'last_modified': "<class 'str'>", 'modified_by': "<class 'str

# Load article and split

In [8]:
articles = list(articles_collection.find({}))
logger.info(f"Found {len(articles)} articles to classify")

count = 0
for article in articles:
    article_id = article.get('article_id', 'Unknown')
    try:        
        # Check if article has been processed
        # logger.debug(f"Processing article {article_id}")
        article_processed = articles_class_collection.find_one({'article_id': article_id, 
                                                            'version': MigrateConfig.MIGRATE_CLASSIFY_ARTICLE_LEVEL_OLLAMA})
        if not article_processed:
            count += 1
            article['classify_instance'] = f"INSTANCE_{count%3}"            
            articles_collection.update_one({'article_id': article_id}, {'$set': {'classify_instance': article['classify_instance']}})
    except Exception as e:
        logger.error(f"Failed to process article {article_id}: {e}")
        continue

2025-06-18 23:50:12.772 | INFO     | __main__:<module>:2 - Found 164204 articles to classify


# Extract Classification Examples

In [10]:
articles = list(articles_class_collection.find({'version': MigrateConfig.MIGRATE_CLASSIFY_ARTICLE_LEVEL_OLLAMA}))[:100]
logger.debug(f"Number of articles to classify: {len(articles)}")

2025-06-19 10:48:45.856 | DEBUG    | __main__:<module>:2 - Number of articles to classify: 100


In [11]:
# Save articles to csv file
import pandas as pd
articles_df = pd.DataFrame(articles)
articles_df.to_csv('articles_classified_by_ollama.csv', index=False)
